In [ ]:
# Plant Analyzer for iPhone/Jupyter (Carnets)
# This version provides a simple UI for plant analysis

import numpy as np
from PIL import Image, ImageDraw, ImageFont
import io
import base64
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt
from collections import Counter

class iPhonePlantAnalyzer:
    def __init__(self):
        self.analysis_results = {}
        
    def analyze_image(self, image_array):
        """Analyze plant image from numpy array"""
        self.pixels = image_array
        self.analysis_results = {}
        
        # Run all analyses
        self.extract_dominant_colors()
        self.analyze_green_health()
        self.detect_discoloration()
        self.analyze_plant_structure()
        
        return self.generate_friendly_report()
    
    def extract_dominant_colors(self, k=5):
        """Extract dominant colors"""
        pixels = self.pixels.reshape(-1, 3)
        
        color_groups = {}
        for pixel in pixels[:5000]:
            pixel_tuple = tuple(int(x) for x in pixel)
            key = tuple((pixel_tuple[i] // 32) * 32 for i in range(3))
            color_groups[key] = color_groups.get(key, 0) + 1
        
        top_colors = sorted(color_groups.items(), key=lambda x: x[1], reverse=True)[:k]
        total = sum(count for _, count in top_colors)
        
        dominant_colors = []
        for color, count in top_colors:
            percentage = (count / total) * 100
            dominant_colors.append({
                'rgb': color,
                'percentage': round(percentage, 2)
            })
        
        self.analysis_results['dominant_colors'] = dominant_colors
        return dominant_colors
    
    def analyze_green_health(self):
        """Analyze plant health based on green color presence"""
        pixels = self.pixels.reshape(-1, 3)
        
        green_pixels = 0
        green_intensities = []
        
        for pixel in pixels[:3000]:
            r, g, b = int(pixel[0]), int(pixel[1]), int(pixel[2])
            if g > r and g > b and g > 50:
                green_pixels += 1
                green_intensities.append(g)
        
        total_sampled = min(3000, len(pixels))
        green_percentage = (green_pixels / total_sampled) * 100
        avg_green_intensity = np.mean(green_intensities) if green_intensities else 0
        
        health_indicators = {
            'green_coverage_percentage': round(green_percentage, 2),
            'avg_green_intensity': round(float(avg_green_intensity), 2),
            'health_score': min(100, round(float((green_percentage * avg_green_intensity) / 255), 2))
        }
        
        self.analysis_results['health_analysis'] = health_indicators
        return health_indicators
    
    def detect_discoloration(self):
        """Detect yellow/brown areas"""
        pixels = self.pixels.reshape(-1, 3)
        
        yellow_count = 0
        brown_count = 0
        
        for pixel in pixels[:3000]:
            r, g, b = int(pixel[0]), int(pixel[1]), int(pixel[2])
            
            if r > 150 and g > 150 and b < 100:
                yellow_count += 1
            
            if 100 < r < 200 and g < r and b < 100:
                brown_count += 1
        
        total_sampled = min(3000, len(pixels))
        yellow_percentage = (yellow_count / total_sampled) * 100
        brown_percentage = (brown_count / total_sampled) * 100
        
        discoloration = {
            'yellow_percentage': round(yellow_percentage, 2),
            'brown_percentage': round(brown_percentage, 2),
            'total_discoloration': round(yellow_percentage + brown_percentage, 2)
        }
        
        self.analysis_results['discoloration_analysis'] = discoloration
        return discoloration
    
    def analyze_plant_structure(self):
        """Analyze basic plant structure"""
        height, width = self.pixels.shape[:2]
        aspect_ratio = width / height
        
        structure_analysis = {
            'image_dimensions': f"{width}x{height}",
            'aspect_ratio': round(float(aspect_ratio), 2),
            'size_category': 'large' if max(width, height) > 1000 else 'medium' if max(width, height) > 500 else 'small'
        }
        
        self.analysis_results['structure_analysis'] = structure_analysis
        return structure_analysis
    
    def generate_friendly_report(self):
        """Generate a user-friendly HTML report"""
        health = self.analysis_results['health_analysis']
        discoloration = self.analysis_results['discoloration_analysis']
        colors = self.analysis_results['dominant_colors']
        structure = self.analysis_results['structure_analysis']
        
        # Determine health status with emoji
        if health['health_score'] > 70:
            health_status = "🌿 Likely Healthy"
            health_color = "green"
        elif health['health_score'] > 40:
            health_status = "⚠️ Moderately Healthy"
            health_color = "orange"
        else:
            health_status = "❌ Potentially Unhealthy"
            health_color = "red"
        
        # Generate recommendations
        recommendations = []
        if health['health_score'] > 70:
            recommendations.append("✅ Your plant appears healthy! Continue current care routine.")
        else:
            recommendations.append("💧 Check your watering schedule")
            recommendations.append("🌱 Consider plant-appropriate fertilizer")
            
        if discoloration['yellow_percentage'] > 10:
            recommendations.append("⚠️ Yellow leaves may indicate overwatering")
            
        if discoloration['brown_percentage'] > 5:
            recommendations.append("🔥 Brown tips could mean too much sun or under-watering")
            
        if health['green_coverage_percentage'] < 30:
            recommendations.append("📷 For better analysis, take a closer photo focused on the plant")
        
        if len(recommendations) == 0:
            recommendations.append("🌿 No specific issues detected. Maintain regular plant care.")
        
        # Create HTML report
        html_report = f"""
        <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; max-width: 600px; margin: 20px auto; padding: 20px; background: #f8f9fa; border-radius: 15px;">
            <h1 style="color: #2e7d32; text-align: center; margin-bottom: 30px;">🌱 Plant Health Analysis</h1>
            
            <div style="background: white; padding: 20px; border-radius: 10px; margin-bottom: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <h2 style="color: #2e7d32; margin-top: 0;">Overall Status: <span style="color: {health_color};">{health_status}</span></h2>
                <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px;">
                    <div style="text-align: center; padding: 10px; background: #e8f5e8; border-radius: 8px;">
                        <h3 style="margin: 0; color: #2e7d32;">Health Score</h3>
                        <p style="font-size: 24px; font-weight: bold; margin: 5px 0; color: {health_color};">{health['health_score']}/100</p>
                    </div>
                    <div style="text-align: center; padding: 10px; background: #e8f5e8; border-radius: 8px;">
                        <h3 style="margin: 0; color: #2e7d32;">Green Coverage</h3>
                        <p style="font-size: 24px; font-weight: bold; margin: 5px 0; color: #2e7d32;">{health['green_coverage_percentage']}%</p>
                    </div>
                </div>
            </div>
            
            <div style="background: white; padding: 20px; border-radius: 10px; margin-bottom: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <h3 style="color: #2e7d32;">📊 Detailed Analysis</h3>
                <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px;">
                    <div><strong>Green Intensity:</strong> {health['avg_green_intensity']}/255</div>
                    <div><strong>Yellow Areas:</strong> {discoloration['yellow_percentage']}%</div>
                    <div><strong>Brown Areas:</strong> {discoloration['brown_percentage']}%</div>
                    <div><strong>Image Size:</strong> {structure['image_dimensions']}</div>
                </div>
            </div>
            
            <div style="background: white; padding: 20px; border-radius: 10px; margin-bottom: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <h3 style="color: #2e7d32;">💡 Care Recommendations</h3>
                <ul style="padding-left: 20px;">
        """
        
        for rec in recommendations:
            html_report += f'<li style="margin-bottom: 8px;">{rec}</li>'
        
        html_report += """
                </ul>
            </div>
            
            <div style="text-align: center; color: #666; font-size: 12px; margin-top: 20px;">
                Analysis completed with Plant Analyzer 🌿
            </div>
        </div>
        """
        
        return html_report

def create_upload_ui():
    """Create a simple upload UI for Jupyter"""
    html_ui = """
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; max-width: 600px; margin: 20px auto; padding: 20px; text-align: center;">
        <h1 style="color: #2e7d32;">🌱 Plant Health Analyzer</h1>
        <p style="color: #666; margin-bottom: 30px;">Upload a photo of your plant to analyze its health</p>
        
        <div style="background: white; padding: 30px; border-radius: 15px; border: 2px dashed #2e7d32;">
            <input type="file" id="plant_upload" accept="image/*" style="display: none;">
            <button onclick="document.getElementById('plant_upload').click()" 
                    style="background: #2e7d32; color: white; border: none; padding: 15px 30px; 
                           border-radius: 25px; font-size: 16px; cursor: pointer;">
                📷 Choose Plant Photo
            </button>
            <p style="color: #999; margin-top: 15px;">or take a new photo with your camera</p>
        </div>
        
        <div id="analysis_result" style="margin-top: 20px;"></div>
    </div>
    
    <script>
    document.getElementById('plant_upload').addEventListener('change', function(event) {
        const file = event.target.files[0];
        if (file) {
            const reader = new FileReader();
            reader.onload = function(e) {
                // Send image data to Python for analysis
                const imageData = e.target.result;
                IPython.notebook.kernel.execute(`analyze_uploaded_image('${imageData}')`);
            };
            reader.readAsDataURL(file);
        }
    });
    </script>
    """
    return HTML(html_ui)

# Global analyzer instance
analyzer = iPhonePlantAnalyzer()

def analyze_uploaded_image(image_data):
    """Analyze uploaded image and display results"""
    try:
        # Clear previous output
        clear_output(wait=True)
        
        # Process image data
        image_data = image_data.split(',')[1]  # Remove data URL prefix
        image_bytes = base64.b64decode(image_data)
        image = Image.open(io.BytesIO(image_bytes))
        
        # Convert to numpy array
        image_array = np.array(image)
        
        # Display original image
        plt.figure(figsize=(8, 6))
        plt.imshow(image_array)
        plt.axis('off')
        plt.title('Your Plant Photo', fontsize=16, pad=20)
        plt.show()
        
        # Analyze image
        print("🔍 Analyzing plant health...")
        html_report = analyzer.analyze_image(image_array)
        
        # Display results
        display(HTML(html_report))
        
        # Show option to analyze another plant
        display(HTML("""
        <div style="text-align: center; margin: 30px;">
            <button onclick="window.location.reload()" 
                    style="background: #2e7d32; color: white; border: none; padding: 12px 24px; 
                           border-radius: 20px; font-size: 14px; cursor: pointer;">
                🌿 Analyze Another Plant
            </button>
        </div>
        """))
        
    except Exception as e:
        display(HTML(f"""
        <div style="background: #ffebee; color: #c62828; padding: 20px; border-radius: 10px; text-align: center;">
            <h3>❌ Error Analyzing Image</h3>
            <p>Please try again with a different photo.</p>
            <p><small>Error: {str(e)}</small></p>
        </div>
        """))

def quick_analysis_demo():
    """Quick demo with sample plant data"""
    display(HTML("""
    <div style="background: #e8f5e8; padding: 20px; border-radius: 10px; text-align: center; margin: 20px 0;">
        <h3>🌱 Quick Demo</h3>
        <p>For testing, here's what a healthy plant analysis looks like:</p>
    </div>
    """))
    
    # Create sample analysis results
    sample_results = {
        'health_analysis': {
            'health_score': 85,
            'green_coverage_percentage': 65.5,
            'avg_green_intensity': 195.2
        },
        'discoloration_analysis': {
            'yellow_percentage': 2.1,
            'brown_percentage': 0.8,
            'total_discoloration': 2.9
        },
        'structure_analysis': {
            'image_dimensions': '800x600'
        }
    }
    
    analyzer.analysis_results = sample_results
    html_report = analyzer.generate_friendly_report()
    display(HTML(html_report))

# Main function to start the analyzer
def start_plant_analyzer():
    """Start the plant analyzer UI"""
    print("🌱 Starting Plant Health Analyzer...")
    display(create_upload_ui())
    
    # Add quick demo option
    display(HTML("""
    <div style="text-align: center; margin: 20px;">
        <button onclick="IPython.notebook.kernel.execute('quick_analysis_demo()')" 
                style="background: #666; color: white; border: none; padding: 10px 20px; 
                       border-radius: 15px; font-size: 12px; cursor: pointer;">
            🧪 See Demo Analysis
        </button>
    </div>
    """))

# Instructions for use
def show_instructions():
    """Show usage instructions"""
    instructions = """
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; 
                max-width: 600px; margin: 20px auto; padding: 20px; background: #e3f2fd; border-radius: 10px;">
        <h2 style="color: #1976d2;">📱 How to Use Plant Analyzer</h2>
        <ol style="text-align: left;">
            <li><strong>Run the analyzer:</strong> Call <code>start_plant_analyzer()</code></li>
            <li><strong>Upload photo:</strong> Click "Choose Plant Photo" to select from your gallery</li>
            <li><strong>Take photo:</strong> On iPhone, you can take a new photo directly</li>
            <li><strong>View results:</strong> Get instant health analysis and care tips</li>
        </ol>
        <p><strong>Tip:</strong> For best results, take a clear photo focused on the plant leaves against a neutral background.</p>
    </div>
    """
    display(HTML(instructions))

# Start here - run this cell to begin
print("Plant Analyzer for iPhone ready!")
print("Run 'start_plant_analyzer()' to begin analyzing your plants!")
print("Run 'show_instructions()' for usage guide.")